# Domain 3: Claude Code Configuration & Workflows
## Claude Certified Architect – Foundations Certification
**Weight: 20% of scored content**

This notebook covers configuring Claude Code for team workflows including CLAUDE.md hierarchies,
custom commands, skills, path-specific rules, plan mode, iterative refinement, and CI/CD integration.

---

## Task 3.1: Configure CLAUDE.md Files with Appropriate Hierarchy, Scoping, and Modular Organization

### The CLAUDE.md Hierarchy

```
~/.claude/CLAUDE.md          ← User-level (personal, NOT shared via version control)
├── .claude/CLAUDE.md        ← Project-level (shared with team)
│   or root CLAUDE.md
├── src/CLAUDE.md            ← Directory-level (applies to src/ and below)
├── src/api/CLAUDE.md        ← Deeper directory-level
└── .claude/rules/           ← Topic-specific rule files (alternative to monolithic CLAUDE.md)
    ├── testing.md
    ├── api-conventions.md
    └── deployment.md
```

### Key Knowledge Points

| Level | Location | Shared? | Use Case |
|---|---|---|---|
| User-level | `~/.claude/CLAUDE.md` | ❌ Personal only | Personal preferences, editor settings |
| Project-level | `.claude/CLAUDE.md` or root `CLAUDE.md` | ✅ Version controlled | Team coding standards, test conventions |
| Directory-level | Subdirectory `CLAUDE.md` files | ✅ Version controlled | Area-specific conventions |
| Rules | `.claude/rules/` | ✅ Version controlled | Topic-specific, optionally path-scoped |

### Common Exam Scenario

> A new team member doesn't receive certain instructions that other team members get.
>
> **Root cause:** Instructions are in `~/.claude/CLAUDE.md` (user-level) instead of
> `.claude/CLAUDE.md` (project-level). User-level configs are not shared via version control.

### Modular Organization with `@import`

```markdown
# CLAUDE.md
@import ./standards/python-style.md
@import ./standards/testing.md
@import ./standards/api-conventions.md
```

Keeps CLAUDE.md modular — selectively include relevant standards per package.

In [ ]:
# === CLAUDE.md HIERARCHY SIMULATION ===

import os

def simulate_claude_md_resolution(working_file):
    """
    Simulates how Claude Code resolves CLAUDE.md configuration.
    
    Key exam point: Configurations are layered.
    User-level → Project-level → Directory-level
    """
    configs = []
    
    # 1. User-level (always loaded, personal only)
    configs.append({
        "source": "~/.claude/CLAUDE.md",
        "scope": "user",
        "shared": False,
        "content": "Personal preferences (not shared with team)"
    })
    
    # 2. Project-level (shared via version control)
    configs.append({
        "source": ".claude/CLAUDE.md",
        "scope": "project",
        "shared": True,
        "content": "Team coding standards, test conventions"
    })
    
    # 3. Rules directory (path-scoped rules)
    configs.append({
        "source": ".claude/rules/testing.md",
        "scope": "path-scoped",
        "shared": True,
        "content": "Testing conventions (loaded when editing test files)",
        "paths": ["**/*.test.*", "**/*.spec.*"]
    })
    
    # 4. Directory-level (if editing in a subdirectory)
    dir_path = os.path.dirname(working_file)
    if dir_path:
        configs.append({
            "source": f"{dir_path}/CLAUDE.md",
            "scope": "directory",
            "shared": True,
            "content": f"Conventions specific to {dir_path}/"
        })
    
    return configs

# Example: What configs load when editing src/api/routes.py?
configs = simulate_claude_md_resolution("src/api/routes.py")
print("Configs loaded when editing src/api/routes.py:")
for c in configs:
    shared = "✅ Shared" if c['shared'] else "❌ Personal"
    print(f"  [{c['scope']}] {c['source']} ({shared})")

### Diagnostic Commands

- `/memory` — Verify which memory files are loaded
- Use to diagnose inconsistent behavior across sessions

---

## Task 3.2: Create and Configure Custom Slash Commands and Skills

### Slash Commands

| Scope | Location | Shared? |
|---|---|---|
| **Project-scoped** | `.claude/commands/` | ✅ Version controlled |
| **User-scoped** | `~/.claude/commands/` | ❌ Personal |

### Skills

Located in `.claude/skills/` with `SKILL.md` files that support frontmatter:

```yaml
---
context: fork          # Run in isolated sub-agent context
allowed-tools:         # Restrict tool access during execution
  - Read
  - Write
argument-hint: "Enter the file path to analyze"  # Prompt for required params
---

## Skill Instructions
Analyze the specified file and generate a comprehensive review...
```

### Key Frontmatter Options

| Option | Purpose | When to Use |
|---|---|---|
| `context: fork` | Isolated sub-agent context | Verbose output, exploratory analysis |
| `allowed-tools` | Restrict tool access | Prevent destructive actions |
| `argument-hint` | Prompt for parameters | When skill needs user input |

### Skills vs CLAUDE.md

| Feature | Skills | CLAUDE.md |
|---|---|---|
| Loading | On-demand invocation | Always loaded |
| Purpose | Task-specific workflows | Universal standards |
| Context | Can be forked (isolated) | Main conversation |

In [ ]:
# === SKILL AND COMMAND CONFIGURATION EXAMPLES ===

import json

# Example project structure for commands and skills
project_structure = {
    ".claude/commands/": {
        "review.md": "# Review Command\nRun the team's standard code review checklist...",
        "test-gen.md": "# Test Generation\nGenerate tests for the specified module...",
    },
    ".claude/skills/": {
        "codebase-analysis/SKILL.md": '''---
context: fork
allowed-tools:
  - Read
  - Grep
  - Glob
argument-hint: "Enter the module path to analyze"
---

Analyze the specified module's architecture, dependencies, 
and test coverage. Return a structured summary.''',
        "refactor/SKILL.md": '''---
context: fork
allowed-tools:
  - Read
  - Write
  - Edit
  - Bash
---

Refactor the specified code following SOLID principles.
Run tests after each change to verify correctness.'''
    }
}

print("Project Command & Skill Structure:")
for directory, files in project_structure.items():
    print(f"\n{directory}")
    for filename, content in files.items():
        preview = content.split('\n')[0][:60]
        print(f"  ├── {filename}: {preview}...")

print("\n\nKey exam point:")
print("  - Commands in .claude/commands/ → shared via version control")
print("  - Commands in ~/.claude/commands/ → personal only")
print("  - context: fork → prevents skill output from polluting main conversation")

---

## Task 3.3: Apply Path-Specific Rules for Conditional Convention Loading

### Path-Scoped Rules with YAML Frontmatter

```yaml
# .claude/rules/terraform.md
---
paths:
  - "terraform/**/*"
---

## Terraform Conventions
- Use `terraform fmt` before committing
- All resources must have tags...
```

### When to Use Path-Specific Rules vs Directory CLAUDE.md

| Approach | Best For |
|---|---|
| **Path-specific rules** (`.claude/rules/`) | Conventions spanning multiple directories (e.g., test files everywhere) |
| **Directory CLAUDE.md** | Conventions for a single directory tree |

### Key Advantage

Path-scoped rules with glob patterns like `**/*.test.tsx` apply to test files
**regardless of directory location** — essential when test files are spread throughout the codebase.

Rules load **only** when editing matching files, reducing irrelevant context and token usage.

In [ ]:
# === PATH-SPECIFIC RULES EXAMPLES ===

import fnmatch

# Define path-scoped rules
rules = [
    {
        "file": ".claude/rules/testing.md",
        "paths": ["**/*.test.*", "**/*.spec.*", "**/tests/**/*"],
        "content": "Use Jest with React Testing Library. Mock external APIs."
    },
    {
        "file": ".claude/rules/terraform.md",
        "paths": ["terraform/**/*", "infra/**/*.tf"],
        "content": "Use terraform fmt. All resources must have tags."
    },
    {
        "file": ".claude/rules/api.md",
        "paths": ["src/api/**/*", "**/routes/**/*"],
        "content": "Use async/await with structured error responses."
    }
]

def get_active_rules(current_file, all_rules):
    """Determine which rules load for the current file being edited."""
    active = []
    for rule in all_rules:
        for pattern in rule["paths"]:
            if fnmatch.fnmatch(current_file, pattern):
                active.append(rule)
                break
    return active

# Test which rules activate for different files
test_files = [
    "src/components/Button.test.tsx",
    "terraform/main.tf",
    "src/api/users/routes.py",
    "src/utils/helpers.py",
    "tests/integration/test_auth.py"
]

for f in test_files:
    active = get_active_rules(f, rules)
    rule_names = [r['file'].split('/')[-1] for r in active] or ["(no path-specific rules)"]
    print(f"  {f}")
    print(f"    → Rules loaded: {', '.join(rule_names)}")

---

## Task 3.4: Determine When to Use Plan Mode vs Direct Execution

### Decision Matrix

| Characteristic | Plan Mode ✅ | Direct Execution ✅ |
|---|---|---|
| **Scope** | Large-scale changes, multiple files | Single file, well-scoped |
| **Approaches** | Multiple valid approaches exist | Clear, obvious approach |
| **Decisions** | Architectural decisions needed | Implementation is straightforward |
| **Risk** | Costly rework if wrong approach | Low risk, easy to undo |
| **Example** | Microservice restructuring, library migration (45+ files) | Bug fix with clear stack trace, adding a validation check |

### The Explore Subagent

- Isolates **verbose discovery output** from the main conversation
- Returns summaries to preserve main conversation context
- Prevents context window exhaustion during multi-phase tasks

### Combined Approach

Plan mode for **investigation** → Direct execution for **implementation**

Example: Plan a library migration (explore codebase, design approach) → Execute the planned approach directly

In [ ]:
# === PLAN MODE vs DIRECT EXECUTION DECISION GUIDE ===

def recommend_mode(task_description):
    """
    Decision guide matching exam expectations.
    """
    plan_indicators = [
        "restructure", "migrate", "redesign", "architect",
        "multiple files", "service boundaries", "refactor",
        "multiple approaches", "large-scale", "microservice"
    ]
    
    direct_indicators = [
        "bug fix", "single file", "add validation", "clear stack trace",
        "typo", "one function", "simple change", "well-understood"
    ]
    
    task_lower = task_description.lower()
    plan_score = sum(1 for kw in plan_indicators if kw in task_lower)
    direct_score = sum(1 for kw in direct_indicators if kw in task_lower)
    
    if plan_score > direct_score:
        return "PLAN MODE", "Complex task with architectural implications"
    elif direct_score > plan_score:
        return "DIRECT EXECUTION", "Well-scoped change with clear approach"
    else:
        return "PLAN MODE (default for ambiguous)", "When in doubt, plan first"

# Exam-style scenarios
scenarios = [
    "Restructure monolithic application into microservices across dozens of files",
    "Fix a single-file bug with a clear stack trace pointing to a null check",
    "Migrate from library A to library B affecting 45+ files",
    "Add a date validation check to one function",
    "Choose between two integration approaches with different infrastructure requirements"
]

for scenario in scenarios:
    mode, reason = recommend_mode(scenario)
    print(f"  → {mode}")
    print(f"    Task: {scenario[:70]}...")
    print(f"    Reason: {reason}\n")

---

## Task 3.5: Apply Iterative Refinement Techniques for Progressive Improvement

### Techniques

| Technique | When to Use |
|---|---|
| **Concrete I/O examples** | Prose descriptions produce inconsistent results |
| **Test-driven iteration** | Write tests first, iterate by sharing failures |
| **Interview pattern** | Unfamiliar domain — let Claude ask questions first |
| **Batch vs sequential fixing** | Interacting issues → single message; independent → sequential |

### Key Insight

Providing 2-3 **concrete input/output examples** is the **most effective** way to communicate
expected transformations when prose descriptions are interpreted inconsistently.

In [ ]:
# === ITERATIVE REFINEMENT PATTERNS ===

# Pattern 1: Concrete I/O Examples
refinement_prompt = '''
Transform customer feedback into structured categories.

## Examples (input → expected output)

Input: "The app crashes when I try to upload photos larger than 5MB"
Output: {"category": "bug", "component": "upload", "severity": "high", 
         "repro_steps": "Upload photo > 5MB"}

Input: "It would be great if you could add dark mode"
Output: {"category": "feature_request", "component": "ui", "severity": "low",
         "repro_steps": null}

Input: "The search is really slow on mobile"
Output: {"category": "performance", "component": "search", "severity": "medium",
         "repro_steps": "Use search on mobile device"}
'''

# Pattern 2: Test-Driven Iteration
tdd_prompt = '''
Here are the failing tests after your last change:

FAIL test_migration.py::test_null_values
  Expected: None
  Got: "" (empty string)
  
FAIL test_migration.py::test_date_format 
  Expected: "2024-03-15"
  Got: "03/15/2024"

Please fix these specific issues.
'''

# Pattern 3: Interview Pattern
interview_prompt = '''
I want to add caching to our API.
Before implementing, please ask me questions about:
- Cache invalidation requirements
- Expected data freshness
- Failure modes to consider
'''

print("Iterative Refinement Patterns:")
print("  1. Concrete I/O Examples — most effective for format consistency")
print("  2. Test-Driven Iteration — share failures to guide improvement")
print("  3. Interview Pattern — surface considerations before implementing")
print("  4. Batch vs Sequential:")
print("     - Interacting issues → fix all in one message")
print("     - Independent issues → fix sequentially")

---

## Task 3.6: Integrate Claude Code into CI/CD Pipelines

### Essential CLI Flags

| Flag | Purpose |
|---|---|
| `-p` / `--print` | Non-interactive mode (prevents input hangs in CI) |
| `--output-format json` | Machine-parseable output |
| `--json-schema` | Enforce structured output schema |

### Key Exam Question

> Pipeline hangs indefinitely when running `claude "Analyze this PR"`
>
> **Answer:** Add the `-p` flag: `claude -p "Analyze this PR"`
>
> Other options (`CLAUDE_HEADLESS=true`, `--batch`, `< /dev/null`) don't exist or don't work.

### CI Best Practices

1. **Session context isolation**: Don't use the same session that generated code to review it
   - A separate review instance catches more issues
2. **Include prior review findings** when re-running after new commits
   - Report only new/unaddressed issues to avoid duplicates
3. **Provide existing test files** in context for test generation
   - Avoids suggesting duplicate scenarios
4. **Document standards in CLAUDE.md** for CI-invoked Claude Code

In [ ]:
# === CI/CD INTEGRATION EXAMPLES ===

# GitHub Actions workflow example
ci_workflow = '''
name: Claude Code Review
on:
  pull_request:
    types: [opened, synchronize]

jobs:
  review:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Run Claude Code Review
        run: |
          # ✅ CORRECT: Use -p flag for non-interactive mode
          claude -p "Review this PR for security issues and bugs" \
            --output-format json \
            --json-schema '{"type":"object","properties":{"findings":{"type":"array","items":{"type":"object","properties":{"file":{"type":"string"},"line":{"type":"integer"},"severity":{"type":"string","enum":["critical","high","medium","low"]},"issue":{"type":"string"},"suggestion":{"type":"string"}}}}}}'
          
          # ❌ WRONG: This will hang!
          # claude "Review this PR"
          
          # ❌ WRONG: These flags don't exist
          # CLAUDE_HEADLESS=true claude "Review this PR"
          # claude --batch "Review this PR"
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
'''

print("CI/CD Integration Key Points:")
print("  ✅ claude -p 'prompt' → non-interactive mode")
print("  ✅ --output-format json → machine-parseable output")
print("  ✅ --json-schema '{...}' → enforce output structure")
print("  ❌ CLAUDE_HEADLESS=true → does not exist")
print("  ❌ --batch flag → does not exist")
print("  ❌ < /dev/null → doesn't properly address Claude Code syntax")
print("\n  ⚠️  Use independent review instances (not the same session that generated code)")

---

## Domain 3 Summary & Exam Preparation

### Key Concepts to Remember

1. **CLAUDE.md Hierarchy**: User-level (personal) → Project-level (shared) → Directory-level
2. **New team member issue?** Check if instructions are in user-level instead of project-level
3. **`@import`** for modular CLAUDE.md organization
4. **`.claude/rules/`** with YAML frontmatter for path-scoped rules
5. **Glob patterns** (e.g., `**/*.test.tsx`) for cross-directory conventions
6. **Skills**: `context: fork` isolates output, `allowed-tools` restricts access
7. **Plan mode**: Complex/architectural tasks; **Direct execution**: Simple/well-scoped
8. **CI/CD**: `-p` flag is the answer; `--output-format json` for structured output
9. **Iterative refinement**: Concrete I/O examples > prose descriptions
10. **Interview pattern**: Let Claude ask questions before implementing in unfamiliar domains

### Sample Exam Question

> Test files are spread throughout the codebase. What's the most maintainable way
> to apply consistent testing conventions?
>
> ✅ `.claude/rules/` with glob patterns `**/*.test.tsx`
>
> ❌ Consolidate in root CLAUDE.md (relies on inference)
>
> ❌ Skills in `.claude/skills/` (requires manual invocation)
>
> ❌ Separate CLAUDE.md per directory (can't handle spread-out files)